# Parte 1 – Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| | |
|---|---|
| **Fecha de extracción** | 25 de octubre de 2026 |
| **Corpus A – Documentos** | 25 imágenes de documentos peruanos de Wikimedia Commons: 8 impresos, 8 fotos con sombras, reflejos o perspectiva y 9 manuscritos |
| **Licencias (documentos)** | 15 dominio público, 2 CC0, 7 CC BY-SA (3.0 y 4.0), 1 CC BY 2.0. Detalle por archivo (URL, autor, licencia) en `datos/fuentes.csv` |
| **Corpus B – Audio** | *(se completa en la sección de ASR)* |
| **OCR** | PaddleOCR 3.7.0, modelos PP-OCRv6 *medium* (detección + reconocimiento), motor PyTorch (`engine="transformers"`) en la GPU de la Mac (MPS) |
| **ASR** | *(se completa en la sección de ASR)* |
| **Entorno** | `.venv` del repositorio del curso (`uv sync --group docai` y `uv sync --group audio`) |

Los archivos originales se guardan en `datos/raw/`, que no se versiona en Git (ver `.gitignore`).
Para recuperarlos basta con ejecutar este notebook: la sección 1 los descarga de nuevo a partir de `datos/fuentes.csv`.


In [1]:
import json
import logging
import os
import time
import warnings
from pathlib import Path

# Estas dos líneas van ANTES de importar los modelos (igual que en la Sesión 10)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import paddleocr
import pandas as pd
import requests
import torch
from paddleocr import PaddleOCR
from PIL import Image, ImageDraw

for nombre in ("paddlex", "paddleocr"):
    logging.getLogger(nombre).setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 80)

DATOS = Path("datos")
RAW_DOCS = DATOS / "raw" / "documentos"
RAW_OCR = RAW_DOCS / "ocr"
RAW_OCR.mkdir(parents=True, exist_ok=True)

DISPOSITIVO = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("PaddleOCR", paddleocr.__version__, "| PyTorch", torch.__version__, "| dispositivo:", DISPOSITIVO)

PaddleOCR 3.7.0 | PyTorch 2.13.0 | dispositivo: mps


## 1 · Documentos: descarga desde Wikimedia Commons

Los 25 documentos se eligieron y revisaron a mano. Cada uno está registrado en `datos/fuentes.csv` con su
URL, autor y licencia. La descarga sigue el patrón de la Sesión 10: un `User-Agent` descriptivo,
pausas de 1,5 s entre solicitudes y no volver a bajar lo que ya está en disco.

- **Imágenes:** se piden a 1600 px de ancho (Commons entrega el tamaño estándar más cercano, entre 1200 y 1920 px).
- **PDF:** se usa la página 1, renderizada a 1280 px, que es el máximo que ofrece Commons para estos archivos.

In [2]:
API = "https://commons.wikimedia.org/w/api.php"
AGENTE = {"User-Agent": "tarea2-ds-python/2026 (https://github.com/isabellaach/python-qlab)"}


def url_imagen(titulo):
    "URL de la versión reducida de un archivo de Commons (página 1 si es un PDF)."
    params = dict(action="query", titles="File:" + titulo, prop="imageinfo", iiprop="url",
                  format="json", formatversion=2)
    if titulo.lower().endswith(".pdf"):
        params["iiurlparam"] = "page1-1280px"
    else:
        params["iiurlwidth"] = 1600
    r = requests.get(API, params=params, headers=AGENTE, timeout=60)
    r.raise_for_status()
    time.sleep(1.5)                                   # cortesía con el servidor
    return r.json()["query"]["pages"][0]["imageinfo"][0]["thumburl"]


def descargar(titulo, destino):
    "Baja un documento de Commons. Si ya está en disco, no hace nada."
    if not destino.exists():
        r = requests.get(url_imagen(titulo), headers=AGENTE, timeout=120)
        r.raise_for_status()
        destino.write_bytes(r.content)
        time.sleep(1.5)
    return destino


fuentes = pd.read_csv(DATOS / "fuentes.csv")
docs = fuentes[fuentes["modalidad"] == "documento"].copy()
docs["ruta"] = [descargar(t, Path(a)) for t, a in zip(docs["titulo"], docs["archivo"])]
docs["tamaño"] = [Image.open(r).size for r in docs["ruta"]]

print(docs["tipo"].value_counts().to_string(), "\n")
docs[["id", "tipo", "titulo", "licencia", "tamaño"]]

tipo
manuscrito    9
impreso       8
foto          8 



,id,tipo,titulo,licencia,tamaño
0,doc_001,impreso,Comunicado oficial N 21 sobre el fallecimiento de 2 personas por covid-19 en...,Public domain,"(893, 1280)"
1,doc_002,impreso,El Comercio - Lima. Perú y Ecuador. 3-3-1842.jpg,Public domain,"(675, 925)"
2,doc_003,impreso,"Carátula de El Montonero (n. 12, año 1834).jpg",Public domain,"(1430, 2239)"
3,doc_004,impreso,Inundación Carhuaz 1844.png,Public domain,"(1376, 1950)"
4,doc_005,impreso,La Tribuna-1-edicion.jpg,Public domain,"(904, 1246)"
5,doc_006,impreso,"Assassination of Antonio Miro Quesada, 1935.jpg",Public domain,"(1174, 1725)"
6,doc_007,impreso,Proclama Don Josef Fernando de Abascal y Sousa (1808).pdf,Public domain,"(1280, 1854)"
7,doc_008,impreso,"4000004161-0016 - Carta de Carlos J. Bachmann. 1914 julio 4, Maldonado; para...",CC0,"(1280, 1682)"
8,doc_009,foto,Placa del Edificio Fénix.jpg,CC BY-SA 4.0,"(1920, 1440)"
9,doc_010,foto,"N°1 Edition Of ""El Peruano"" Newspaper.jpg",CC BY-SA 4.0,"(1920, 1280)"


## 2 · OCR con PaddleOCR

Usamos la misma configuración de la clase y **no la ajustamos según los resultados**. Es la línea base de la Parte 2:

- PP-OCRv6 *medium* para detección y reconocimiento, con el motor PyTorch en la GPU de la Mac.
- Sin clasificador de orientación de página, sin "planchado" de hojas dobladas y sin orientación por línea.
- **Sin preprocesamiento:** las imágenes entran tal como vienen de Commons (color, sin binarizar ni enderezar).
  Así vemos cuánto pierde el modelo con las fotos inclinadas y los manuscritos.

Por cada documento guardamos un respaldo crudo en `datos/raw/documentos/ocr/`:
- un `.json` con cada línea leída, su confianza y su caja;
- una imagen `_cajas.jpg` con las cajas dibujadas, verdes si la confianza es alta, amarillas si es media y rojas si es baja.

Si el `.json` ya existe, no se vuelve a ejecutar el OCR.

In [3]:
OPCIONES = dict(
    use_doc_orientation_classify=False,   # no girar la página completa
    use_doc_unwarping=False,              # no "planchar" hojas dobladas
    use_textline_orientation=False,       # no girar cada línea
)


def redes(obj, _prof=0, _vistos=None):
    "Busca, dentro de un modelo de PaddleOCR, las redes neuronales de PyTorch (función de la Sesión 10)."
    _vistos = set() if _vistos is None else _vistos
    if id(obj) in _vistos or _prof > 7:
        return
    _vistos.add(id(obj))
    if isinstance(obj, torch.nn.Module):
        yield obj
        return
    for hijo in (vars(obj).values() if hasattr(obj, "__dict__") else []):
        for x in (hijo.values() if isinstance(hijo, dict) else [hijo]):
            if not isinstance(x, (str, int, float, bool, type(None), list, tuple, set)):
                yield from redes(x, _prof + 1, _vistos)


ocr = PaddleOCR(engine="transformers", **OPCIONES)
for red in redes(ocr):
    red.to(DISPOSITIVO)
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (PP-OCRv6 medium det+rec, engine=transformers, {DISPOSITIVO}, sin preprocesamiento)"
MODELO_OCR

Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

'PaddleOCR 3.7.0 (PP-OCRv6 medium det+rec, engine=transformers, mps, sin preprocesamiento)'

In [4]:
COLORES = {"alta": (12, 163, 12), "media": (250, 178, 25), "baja": (208, 59, 59)}


def nivel(confianza):
    return "alta" if confianza >= 0.9 else ("media" if confianza >= 0.7 else "baja")


def leer(ruta):
    "OCR de una imagen: devuelve una lista de líneas (texto, confianza y caja) y los segundos que tardó."
    t0 = time.perf_counter()
    r = ocr.predict(str(ruta))[0]
    segundos = time.perf_counter() - t0
    cajas = np.asarray(r["rec_boxes"]).reshape(-1, 4).tolist()
    lineas = [{"texto": t, "confianza": round(float(c), 4), "caja": [int(v) for v in b]}
              for t, c, b in zip(r["rec_texts"], r["rec_scores"], cajas)]
    return lineas, segundos


def dibujar(ruta, lineas, destino):
    "Guarda una copia de la imagen con las cajas de cada línea, coloreadas por confianza."
    im = Image.open(ruta).convert("RGB")
    d = ImageDraw.Draw(im)
    for l in lineas:
        d.rectangle(l["caja"], outline=COLORES[nivel(l["confianza"])], width=3)
    im.save(destino, quality=85)


def ocr_documento(id_, ruta):
    "Lee un documento y guarda el respaldo crudo. Si ya fue leído, reutiliza el respaldo."
    crudo = RAW_OCR / f"{id_}.json"
    if crudo.exists():
        return json.loads(crudo.read_text(encoding="utf-8"))
    lineas, segundos = leer(ruta)
    salida = {"id": id_, "archivo": str(ruta), "modelo_ocr": MODELO_OCR,
              "segundos": round(segundos, 2), "lineas": lineas}
    crudo.write_text(json.dumps(salida, ensure_ascii=False, indent=1), encoding="utf-8")
    dibujar(ruta, lineas, RAW_OCR / f"{id_}_cajas.jpg")
    return salida


leer(docs["ruta"].iloc[0])        # primera pasada de calentamiento: carga los modelos en la GPU
resultados = {id_: ocr_documento(id_, ruta) for id_, ruta in zip(docs["id"], docs["ruta"])}
print("Documentos leídos:", len(resultados))

Documentos leídos: 25


In [5]:
def resumen(r):
    conf = [l["confianza"] for l in r["lineas"]]
    texto = "\n".join(l["texto"] for l in r["lineas"])
    return {"id": r["id"], "lineas": len(conf), "caracteres": len(texto),
            "confianza_media": round(float(np.mean(conf)), 3) if conf else None,
            "lineas_baja_conf": sum(c < 0.7 for c in conf), "segundos": r["segundos"], "texto_ocr": texto}


ocr_docs = docs[["id", "tipo"]].merge(pd.DataFrame([resumen(r) for r in resultados.values()]), on="id")
print("Tiempo total de OCR:", round(ocr_docs["segundos"].sum(), 1), "s\n")
print(ocr_docs.groupby("tipo")[["lineas", "caracteres", "confianza_media", "segundos"]].mean().round(2).to_string(), "\n")
ocr_docs.drop(columns="texto_ocr")

Tiempo total de OCR: 125.0 s

            lineas  caracteres  confianza_media  segundos
tipo                                                     
foto         36.50     1238.50             0.87      4.80
impreso      98.50     3914.00             0.96      6.13
manuscrito   22.67      583.22             0.71      4.18 



,id,tipo,lineas,caracteres,confianza_media,lineas_baja_conf,segundos
0,doc_001,impreso,23,955,0.959,1,0.97
1,doc_002,impreso,121,5134,0.963,0,5.90
2,doc_003,impreso,76,2597,0.988,0,4.55
3,doc_004,impreso,136,5128,0.990,0,6.92
4,doc_005,impreso,237,7028,0.902,8,13.19
5,doc_006,impreso,146,8091,0.961,2,11.54
6,doc_007,impreso,24,944,0.980,0,2.27
7,doc_008,impreso,25,1435,0.947,1,3.71
8,doc_009,foto,15,778,0.953,1,3.51
9,doc_010,foto,96,3334,0.560,82,8.94


Un vistazo a un documento de cada tipo: el inicio del texto leído y su confianza media.

In [6]:
for tipo in ["impreso", "foto", "manuscrito"]:
    fila = ocr_docs[ocr_docs["tipo"] == tipo].iloc[0]
    print(f"── {fila['id']} ({tipo}, confianza media {fila['confianza_media']}) ──")
    print(fila["texto_ocr"][:400], "\n")

── doc_001 (impreso, confianza media 0.959) ──
PERÚ
Ministerio
Oficina General
de Salud
de Comunicaciones
COMUNICADO OFICIAL DE PRENSA N°21
El Ministerio de Salud lamenta informar el sensible fallecimiento de
dos personas por infección por COVID-19.
Se trata de un varón de 47 años con antecedentes de viaje a España,
que presentaba asma bronquial y obesidad. Fue internado en la
unidad de cuidados intensivos del hospital 2 de Mayo, por presentar 

── doc_009 (foto, confianza media 0.953) ──
EDIFICIOFENIX
EL EDIFICIO FENIX FVE CONSTRVIDO EN 1934 POR LA FIRMA FRED T. LEY
CIA. LTD. POR ENCARCO DE LA COMPAÑIA DE SEGVROS LA FENIX PERVANA
PARA ALBERGAR SV SEDE SOCIAL.
EL ENCARCO CONSISTIO EN ANADIR DOS PISOS Y VNA AZOTEA DE
DEPARTAMENTOS A VNA FINCA EXISTENTE DE VN SOLO PISO DONDE
FVNCIONABAN ALMACENES Y COMERCIOS EN RENTA. EN LA JEGVNDA PLANTA
FVNCIONARON LAS OFICINAS DE LA COMPANÍA DE SE 

── doc_017 (manuscrito, confianza media 0.658) ──
Fugill, 29 de pili 917
ot. oni lai Egut
SAannnco.
hom